In [ ]:
# Imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import DatasetFolder
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import numpy as np
import gc

from generative.networks.nets import DiffusionModelUNet
from generative.networks.schedulers import DDPMScheduler, DDIMScheduler
from generative.inferers import DiffusionInferer
from torch.amp import autocast, GradScaler

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# ============================================
# CONFIGURATION - Dental OPG Xray
# ============================================
IMAGE_SIZE = 64           # Reduced for memory efficiency
BATCH_SIZE = 8            # Smaller batch for GPU memory
NUM_EPOCHS = 100
LEARNING_RATE = 1e-4
NUM_TRAIN_TIMESTEPS = 1000

# Paths
DATA_DIR = Path(r"c:\Pranav Aditya\MP01")
DENTAL_DIR = DATA_DIR / "extracted_data" / "CT SCANS" / "Dental OPG Xray Dataset" / "Dental OPG Xray Dataset" / "data"
MODEL_SAVE_PATH = DATA_DIR / "dental_opg_xray_model.pth"

print(f"📁 Dataset: {DENTAL_DIR}")
print(f"💾 Model will be saved to: {MODEL_SAVE_PATH}")
print(f"🖼️  Config: {IMAGE_SIZE}x{IMAGE_SIZE} RGB, Batch {BATCH_SIZE}")

In [ ]:
# ============================================
# DATA LOADING - RGB COLORED IMAGES
# ============================================
IMG_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif')

# Transforms for RGB images
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Load dataset
dataset = DatasetFolder(
    root=str(DENTAL_DIR),
    loader=lambda p: Image.open(p).convert('RGB'),
    extensions=IMG_EXTENSIONS,
    transform=train_transforms
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

print(f"✅ Loaded {len(dataset)} images at {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"📊 Classes: {dataset.classes}")
print(f"📦 Batches per epoch: {len(dataloader)}")

In [ ]:
# ============================================
# VISUALIZE SAMPLE IMAGES
# ============================================
sample_batch = next(iter(dataloader))
sample_images = sample_batch[0][:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    img = sample_images[i].permute(1, 2, 0).numpy()
    img = (img * 0.5 + 0.5)  # Denormalize
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Sample {i+1}')

plt.suptitle('🦷 Dental OPG Xray Dataset - Sample Images (RGB)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CREATE MODEL FOR 8GB VRAM
# ============================================
# Clear memory
gc.collect()
torch.cuda.empty_cache()

print(f"💾 GPU before model: {torch.cuda.memory_allocated()/1e9:.2f} GB")

model = DiffusionModelUNet(
    spatial_dims=2,
    in_channels=3,
    out_channels=3,
    num_channels=(64, 128, 256, 256),
    attention_levels=(False, False, True, True),
    num_res_blocks=2,
    num_head_channels=64,
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"🧠 Model Parameters: {num_params:,}")

scheduler = DDPMScheduler(
    num_train_timesteps=NUM_TRAIN_TIMESTEPS,
    schedule="linear_beta",
)
inferer = DiffusionInferer(scheduler)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"✅ Model created!")
print(f"💾 GPU after model: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ============================================
# TRAINING LOOP - WITH MIXED PRECISION
# ============================================
print("="*60)
print("🦷 DENTAL OPG XRAY MODEL TRAINING")
print("="*60)
print(f"📊 Dataset: {len(dataset)} images")
print(f"🖼️  Resolution: {IMAGE_SIZE}x{IMAGE_SIZE} RGB")
print(f"📦 Batch size: {BATCH_SIZE}")
print(f"🔄 Epochs: {NUM_EPOCHS}")
print(f"🖥️  Device: {device}")
print(f"⚡ Using Mixed Precision (FP16)")
print("="*60)

scaler = GradScaler('cuda')
losses = []
best_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in pbar:
        images = batch[0].to(device)
        
        optimizer.zero_grad()
        
        timesteps = torch.randint(0, NUM_TRAIN_TIMESTEPS, (images.shape[0],)).to(device)
        noise = torch.randn_like(images)
        noisy_images = scheduler.add_noise(images, noise, timesteps)
        
        with autocast('cuda'):
            noise_pred = model(noisy_images, timesteps=timesteps)
            loss = nn.functional.mse_loss(noise_pred, noise)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    epoch_loss = total_loss / len(dataloader)
    losses.append(epoch_loss)
    lr_scheduler.step()
    
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {epoch_loss:.6f} | Best: {best_loss:.6f}")
    
    # Generate samples every 25 epochs
    if (epoch + 1) % 25 == 0:
        model.eval()
        gen_scheduler = DDIMScheduler(num_train_timesteps=NUM_TRAIN_TIMESTEPS, schedule="linear_beta")
        gen_scheduler.set_timesteps(50)
        
        with torch.no_grad():
            noise = torch.randn(4, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
            image = noise
            for t in gen_scheduler.timesteps:
                with autocast('cuda'):
                    out = model(image, timesteps=torch.tensor([t]*4).to(device))
                image, _ = gen_scheduler.step(out, t, image)
            samples = image.cpu()
        
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        for i, ax in enumerate(axes):
            img = samples[i].permute(1, 2, 0).numpy()
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            ax.imshow(img)
            ax.axis('off')
        plt.suptitle(f'Epoch {epoch+1} - Generated Samples')
        plt.tight_layout()
        plt.show()
        
        torch.cuda.empty_cache()

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print(f"💾 Model saved: {MODEL_SAVE_PATH}")
print(f"📉 Best Loss: {best_loss:.6f}")
print("="*60)

In [ ]:
# ============================================
# TRAINING VISUALIZATION
# ============================================
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(losses, 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(losses, 'b-', linewidth=2)
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss (log scale)')
plt.title('Training Loss (Log Scale)')
plt.grid(True, alpha=0.3)

plt.suptitle('Dental OPG Xray Model Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# FINAL GENERATION - HIGH QUALITY
# ============================================
print("Generating high-quality dental OPG xray images...")

model.eval()
gen_scheduler = DDIMScheduler(
    num_train_timesteps=NUM_TRAIN_TIMESTEPS,
    schedule="linear_beta",
)
gen_scheduler.set_timesteps(200)

with torch.no_grad():
    noise = torch.randn(8, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
    image = noise
    
    for t in tqdm(gen_scheduler.timesteps, desc="Generating"):
        model_output = model(image, timesteps=torch.tensor([t] * 8).to(device))
        image, _ = gen_scheduler.step(model_output, t, image)
    
    final_images = image.cpu()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    img = final_images[i].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Generated #{i+1}')

plt.suptitle('AI-Generated Dental OPG Xray Images (64x64 RGB)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✅ Generation complete!")